# Silver guests

Cleaned guest table from `samples.wanderbricks.users` with PII masked: `email` is replaced by a SHA-256 `email_hash` and `name` is dropped. Writes `{catalog}.{schema}.silver_guests` (full refresh). Used to show the `pii-management` skill (silver masking rules); not part of the booking revenue job.

In [0]:
# Silver guests: PII-masked copy of samples.wanderbricks.users
# Target: {catalog}.{schema}.silver_guests (one row per user_id, full refresh)
dbutils.widgets.text("catalog", "genie_code_workshop")
dbutils.widgets.text("schema", "asml_demo")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
target = f"{catalog}.{schema}.silver_guests"
print(target)

In [0]:
from pyspark.sql import functions as F, Window

EMAIL_PATTERN = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

users = spark.table("samples.wanderbricks.users").filter(F.col("user_id").isNotNull())

email_norm = F.lower(F.trim(F.col("email")))
email_missing = F.col("email").isNull() | (F.trim(F.col("email")) == "")

df = (
    users
    # PII masking (pii-management, silver rule): hash email, drop name
    .withColumn("email_hash", F.when(email_missing, None).otherwise(F.sha2(email_norm, 256)))
    .withColumn("email_valid", ~email_missing & email_norm.rlike(EMAIL_PATTERN))
    .withColumn("email_users", F.count("*").over(Window.partitionBy("email_hash")))
    .withColumn(
        "data_quality_flag",
        F.when(email_missing, "MISSING_EMAIL")
        .when(~F.col("email_valid"), "INVALID_EMAIL")
        .when(F.col("email_users") > 1, "DUPLICATE_EMAIL")
        .otherwise("CLEAN"),
    )
    .withColumnRenamed("created_at", "account_created_at")
    .withColumn("audit_timestamp", F.current_timestamp())
    .withColumn("source_system", F.lit("samples.wanderbricks"))
    # identifiers -> dimensions -> derived -> data_quality_flag -> audit columns LAST
    .select(
        "user_id",
        "country", "user_type", "is_business", "company_name", "account_created_at",
        "email_hash",
        "data_quality_flag",
        "audit_timestamp", "source_system",
    )
)

if df.limit(1).count() == 0:
    raise ValueError(f"No rows produced for {target}")

In [0]:
# Full refresh: re-runs replace the table, never duplicate rows
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target)
print(f"Wrote {spark.table(target).count()} rows to {target}")

In [0]:
# DDL per table-governance + pii-management; re-applied on every run because overwriteSchema resets comments
spark.sql(f"""COMMENT ON TABLE {target} IS
  'Cleaned and validated guests with masked email and data quality flag from samples.wanderbricks.users. Name dropped, email replaced by SHA-256 hash - CONTAINS PII: email_hash'""")

spark.sql(f"""ALTER TABLE {target} SET TBLPROPERTIES (
  'quality' = 'silver',
  'domain' = 'guest_experience',
  'contains_pii' = 'true',
  'pii_columns' = 'email_hash',
  'delta.enableChangeDataFeed' = 'true')""")

column_comments = {
    "user_id": "Primary key. References samples.wanderbricks.users.user_id",
    "country": "Country of residence, from samples.wanderbricks.users",
    "user_type": "Type of user: individual or business",
    "is_business": "True if the user is a business",
    "company_name": "Company name for business users, null for individuals",
    "account_created_at": "Account creation timestamp, from samples.wanderbricks.users.created_at",
    "email_hash": "[PII: EMAIL - HIGH, masked] SHA-256 of lower(trim(email)); null when email is missing. Pseudonymous join key, raw email not stored",
    "data_quality_flag": "CLEAN, MISSING_EMAIL, INVALID_EMAIL (fails basic address pattern) or DUPLICATE_EMAIL (same email_hash on more than one user_id)",
    "audit_timestamp": "Job execution timestamp",
    "source_system": "Upstream source system identifier",
}
for col, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {target} ALTER COLUMN {col} COMMENT '{comment}'")

# Governed tag policies in this account reject 'pii' (allows only 'salary') and 'domain' (no allowed values),
# so those are recorded in TBLPROPERTIES (contains_pii, domain) instead
spark.sql(f"""ALTER TABLE {target} SET TAGS (
  'quality' = 'silver', 'data_classification' = 'confidential', 'sla' = 'daily')""")

In [0]:
# Copy of asml-workshop/skills/job-standards/scripts/check_table_standards.py (metadata reads only)
import re

LAYER_PREFIXES = ("bronze_", "silver_", "gold_")
REQUIRED_PROPS = ("quality", "domain")
AUDIT_COLUMNS = ["audit_timestamp", "source_system"]
NAME_PATTERN = re.compile(r"^[a-z][a-z0-9_]*$")

def check_table_standards(full_name: str) -> list:
    failures = []
    table = full_name.split(".")[2]
    if not NAME_PATTERN.match(table):
        failures.append(f"Table name '{table}' is not lowercase_snake_case")
    if not table.startswith(LAYER_PREFIXES):
        failures.append(f"Table name '{table}' has no layer prefix {LAYER_PREFIXES}")
    if not spark.sql(f"DESCRIBE DETAIL {full_name}").first()["description"]:
        failures.append("Missing table COMMENT")
    props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {full_name}").collect()}
    failures += [f"Missing TBLPROPERTY '{p}'" for p in REQUIRED_PROPS if p not in props]
    if props.get("contains_pii") == "true" and not props.get("pii_columns"):
        failures.append("contains_pii=true but 'pii_columns' is not set")
    fields = spark.table(full_name).schema.fields
    if [f.name for f in fields][-2:] != AUDIT_COLUMNS:
        failures.append(f"Last two columns must be {AUDIT_COLUMNS}")
    undocumented = [f.name for f in fields if not f.metadata.get("comment")]
    if undocumented:
        failures.append(f"Columns without COMMENT: {undocumented}")
    print(f"[{'PASS' if not failures else f'FAIL ({len(failures)})'}] {full_name}")
    for f in failures:
        print(f"  - {f}")
    return failures

failures = check_table_standards(target)

# pii-management silver rule: no raw email, phone or name columns
raw_pii = [c for c in spark.table(target).columns if c in ("email", "name", "phone", "first_name", "last_name", "full_name")]
if raw_pii:
    failures.append(f"Raw PII columns present in silver: {raw_pii}")

# Primary key must be unique
dupes = spark.table(target).groupBy("user_id").count().filter("count > 1").count()
if dupes:
    failures.append(f"{dupes} duplicate user_id rows")

if failures:
    raise AssertionError(f"{target} does not meet standards: {failures}")

display(spark.table(target).groupBy("data_quality_flag").count().orderBy("data_quality_flag"))